In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

folder_path = r"C:\Users\brian\OneDrive\Trading\Market Making\data\runs\run_20260829_063459"
snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))
snapshots

,ts,trade_latency,depth_latency,exchange_latency,symbol,mid,mid_tick,microprice,microprice_dev,microprice_error,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1787985301844,0,40,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
1,1787985301944,0,42,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
2,1787985304644,0,50,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
3,1787985305444,0,39,0,PEPEUSDT,0.000004,363,0.000004,-2.809600e-10,2.809600e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
4,1787985305744,0,40,0,PEPEUSDT,0.000004,363,0.000004,-2.818604e-10,2.818604e-10,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23085,1787994879244,38,39,0,PEPEUSDT,0.000004,362,0.000004,2.557403e-09,-2.557403e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN
23086,1787994879844,38,39,0,PEPEUSDT,0.000004,362,0.000004,2.526454e-09,-2.526454e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN
23087,1787994880944,38,44,0,PEPEUSDT,0.000004,362,0.000004,2.525616e-09,-2.525616e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN
23088,1787994881245,38,40,0,PEPEUSDT,0.000004,362,0.000004,2.525616e-09,-2.525616e-09,...,0.0,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN


In [3]:
snapshots.columns


Index(['ts', 'trade_latency', 'depth_latency', 'exchange_latency', 'symbol',
       'mid', 'mid_tick', 'microprice', 'microprice_dev', 'microprice_error',
       'spread', 'best_bid', 'best_ask', 'best_bid_tick', 'best_ask_tick',
       'order_imbalance', 'trade_imbalance', 'volatility', 'queue_ahead_bid',
       'queue_ahead_ask', 'inventory', 'realized_pnl', 'unrealized_pnl',
       'total_pnl', 'fees_paid', 'equity', 'fair', 'skew', 'struct_delta',
       'micro_signal_delta', 'residual_delta', 'reservation', 'regime',
       'regime_id', 'regime_prob', 'alpha_order_imb', 'alpha_trade_imb',
       'alpha_struct', 'k0', 'spread_multiplier', 'inventory_target',
       'residual_signal_quality', 'tox', 'k1', 'k2', 'my_bid', 'my_ask',
       'my_bid_tick', 'my_ask_tick', 'bid_distance_touch',
       'ask_distance_touch', 'bid_distance_spread', 'ask_distance_spread',
       'bid_delta', 'ask_delta', 'quote_churn', 'future_mid_100ms',
       'future_return_100ms', 'future_mid_500ms', 'fut

In [11]:
"""
Key Research Questions

This project is designed to investigate:

Which regimes favor passive liquidity provision? medium frequency trends (1000ms rolling window)

"""

# STEP 1 — Load raw data
df = snapshots
df["ts"] = pd.to_datetime(df["ts"], unit="ms")
df = df.set_index("ts")

# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

"""
2. Choose regime window (critical design choice)
Start simple:
"""

# feature_cols = [
#     "volatility",
#     "spread",
#     "order_imbalance",
#     "trade_imbalance",
#     "quote_churn",
#     "inventory",
#     "inventory_vol",
#     "microprice_error"
# ]

feature_cols = [
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    # "quote_churn", # use only market features, not strategy features
    # "inventory",
    # "inventory_vol",
    "microprice_error"
]

WINDOW = "1s"   # later try 2s, 5s

regime_df = pd.DataFrame()

regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()
regime_df["spread"] = df["spread"].rolling(WINDOW).mean()
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()
# regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()
# regime_df["inventory"] = df["inventory"].rolling(WINDOW).mean()
# regime_df["inventory_vol"] = df["inventory"].rolling(WINDOW).std()
regime_df["microprice_error"] = (df["mid"] - df["microprice"]).rolling(WINDOW).mean()

regime_df = regime_df.dropna()
regime_df

,volatility,spread,order_imbalance,trade_imbalance,microprice_error
ts,,,,,
2026-08-29 06:35:05.444,0.0,1.000000e-08,-0.057351,0.000000,2.867563e-10
2026-08-29 06:35:05.744,0.0,1.000000e-08,-0.056282,0.000000,2.814102e-10
2026-08-29 06:35:05.844,0.0,1.000000e-08,-0.056312,0.000000,2.815603e-10
2026-08-29 06:35:07.544,0.0,1.000000e-08,-0.056482,0.000000,2.824123e-10
2026-08-29 06:35:07.844,0.0,1.000000e-08,-0.056482,0.000000,2.824123e-10
...,...,...,...,...,...
2026-08-29 09:14:38.544,0.0,1.000000e-08,0.512884,0.520148,-2.564419e-09
2026-08-29 09:14:38.944,0.0,1.000000e-08,0.512831,0.520148,-2.564153e-09
2026-08-29 09:14:39.244,0.0,1.000000e-08,0.512508,0.520148,-2.562540e-09


In [5]:
# STEP 3 — Train regime model
scaler = StandardScaler()

X = regime_df[feature_cols].values
X_scaled = scaler.fit_transform(X)

n_regimes = 3  # start small: 2–5 max

model = GaussianMixture(
    n_components=n_regimes,
    covariance_type="full",
    random_state=42
)

regime_df["regime"] = model.fit_predict(X_scaled)

In [6]:
eval_df = df.copy()

times = eval_df.index          # DatetimeIndex
mid = eval_df["mid"].values

horizon_ms = 1000
HORIZON = pd.Timedelta(milliseconds=horizon_ms)

future_return = np.full(len(df), np.nan)
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df)):

    target_time = times[i] + HORIZON

    # first observation at or after t + 1000ms
    j = times.searchsorted(target_time)

    if j >= len(df):
        continue

    p0 = mid[i]
    p1 = mid[j]

    # future window [i, j]
    window = mid[i:j+1]

    # Need at least 2 observations
    if len(window) < 2:
        continue

    # 1. Future return
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility over next 1000ms
    returns = np.diff(window) / window[:-1]
    future_volatility[i] = np.std(returns)

    # 3. Future direction
    # If result ≈ +1
    # almost always up moves after this regime
    # strong bullish bias
    # If result ≈ -1
    # almost always down moves after this regime
    # bearish bias
    # If result ≈ 0
    # no directional bias
    # pure noise / mean reversion / stable
    future_direction[i] = np.sign(p1 - p0)

eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction

eval_df = eval_df.dropna(subset=[ "future_return", "future_volatility", "future_direction"])
eval_df

,trade_latency,depth_latency,exchange_latency,symbol,mid,mid_tick,microprice,microprice_dev,microprice_error,spread,...,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms,future_return,future_volatility,future_direction
ts,,,,,,,,,,,,,,,,,,,,,
2026-08-29 06:35:01.844,0,40,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.0,0.0,0.0
2026-08-29 06:35:01.944,0,42,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.0,0.0,0.0
2026-08-29 06:35:04.644,0,50,0,PEPEUSDT,0.000004,363,0.000004,-2.925525e-10,2.925525e-10,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.0,0.0,0.0
2026-08-29 06:35:05.444,0,39,0,PEPEUSDT,0.000004,363,0.000004,-2.809600e-10,2.809600e-10,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.0,0.0,0.0
2026-08-29 06:35:05.744,0,40,0,PEPEUSDT,0.000004,363,0.000004,-2.818604e-10,2.818604e-10,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,0.000004,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-29 09:14:38.944,38,40,0,PEPEUSDT,0.000004,362,0.000004,2.563622e-09,-2.563622e-09,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN,0.0,0.0,0.0
2026-08-29 09:14:39.244,38,39,0,PEPEUSDT,0.000004,362,0.000004,2.557403e-09,-2.557403e-09,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN,0.0,0.0,0.0
2026-08-29 09:14:39.844,38,39,0,PEPEUSDT,0.000004,362,0.000004,2.526454e-09,-2.526454e-09,1.000000e-08,...,0.0,0.000004,0.0,0.000004,0.0,NaN,NaN,0.0,0.0,0.0


In [7]:
# STEP 5 — ALIGN BOTH DATASETS

# Now regime + outcome are aligned.

final = regime_df.merge(
    eval_df[["future_return", "future_volatility", "future_direction"]],
    left_index=True,
    right_index=True,
    how="inner"
)

# STEP 6 — ANALYZE REGIMES
regime_outcomes = final.groupby("regime").agg({
    "future_return": "mean",
    "future_volatility": "mean",
    "future_direction": "mean"
})

z = final.copy()

for col in feature_cols:
    z[col] = (z[col] - z[col].mean()) / z[col].std()

regime_profile = (
    z.groupby("regime")[feature_cols]
    .mean()
    .round(2)
)

full_profile = pd.DataFrame(regime_profile.join(regime_outcomes))
full_profile

,volatility,spread,order_imbalance,trade_imbalance,microprice_error,future_return,future_volatility,future_direction
regime,,,,,,,,
0,-0.16,0.00,-0.36,0.04,0.36,-0.000026,0.000010,-0.009608
1,-0.16,0.00,1.75,-0.16,-1.75,0.000109,0.000038,0.039485
2,5.84,0.76,-0.18,-0.26,0.18,-0.000029,0.000088,-0.010714


In [9]:
def export_gmm(model_name, gmm, scaler, horizon_ms, feature_cols, regime_labels):
    K = gmm.n_components

    means = gmm.means_

    covs = gmm.covariances_
    precisions = gmm.precisions_  # inverse covariance (what you want)

    log_weights = np.log(gmm.weights_)

    # log determinant of covariance
    log_det = np.array([
        np.log(np.linalg.det(covs[k]))
        for k in range(K)
    ])

    artifact = {
        # GMM
        "means": means.tolist(),
        "cov_inv": precisions.tolist(),
        "log_det_cov": log_det.tolist(),
        "log_weights": log_weights.tolist(),

        # scaler (CRITICAL)
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),

        # metadata
        "model_name": model_name,
        "target": "detect_regime",
        "n_regimes": K,
        "horizon_ms": horizon_ms,
        "feature_cols": feature_cols,
        "regime_labels": [
            regime_labels[i] for i in range(K)
        ]
    }

    with open(f"data/{model_name}.json", "w") as f:
        json.dump(artifact, f)

    print(f"['data/{model_name}.json']")

In [ ]:
regime_labels = {
    0: "low_vol",
    1: "directional",
    2: "high_vol",
}

export_gmm(model_name="regime_model_pepe",
           gmm=model,
           scaler=scaler,
           horizon_ms=horizon_ms,
           feature_cols=feature_cols,
           regime_labels=regime_labels)

# artifact = {
#     "scaler": scaler,
#     "model": model,
#     "feature_cols": feature_cols,
#     "n_regimes": n_regimes,
#     "window": WINDOW,
#     "horizon_ms": 1000,
#     "regime_labels": regime_labels
# }

# joblib.dump(artifact, "data/regime_model_pepe.pkl")

['data/regime_model_pepe.json']


['data/regime_model_pepe.pkl']